In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import time  # Import time module
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout
import csv
import statistics
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)  # Prevent truncation


2025-08-28 15:56:39.353861: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-28 15:56:39.362588: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-28 15:56:39.420912: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-28 15:56:39.464445: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756376799.511365 2112918 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756376799.52

In [2]:
# Load dataset
data = pd.read_csv('/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Final_Data_Table/merged_all_1_to_40.csv')
# video_df= pd.read_csv('/home/rounak/CODE/Third_quadrant_prediction orgnl/pd_ML/video_flag.csv')


In [7]:
data.columns
print(data.columns.to_list())
for col in data.columns:
    print(col)

['GSR_mean', 'GSR_variance', 'HR_mean', 'HR_variance', '75_percentile_GSR', '75_percentile_HR', 'GSRmean_persen_diff', 'HRmean_persent_diff', 'GSRmean_diff', 'HRmean_diff', 'valence_acc_video', 'arousal_acc_video', 'P_id', 'video_id', 'Score', 'time', 'valence', 'arousal', 'videoID', 'start_time', 'end_time', 'probe', 'prev_window', 'start_time_sec', 'end_time_sec', 'num_samples', 'pupil_diameter_mean', 'pupil_diameter_std', 'diameter_3d_mean', 'diameter_3d_std', 'sphere_radius_mean', 'circle_3d_radius_mean', 'ellipse_axis_a_mean', 'ellipse_axis_b_mean', 'ellipse_angle_mean', 'ellipse_angle_std', 'confidence_x_mean', 'confidence_y_mean', 'confidence_x_std', 'confidence_y_std', 'blink_count', 'gaze_movement_mean', 'gaze_movement_std', 'gaze_dir_variability', 'eye_center_movement_mean', 'eye_center_movement_std', 'missing_value_ratio', 'start_time_ms', 'end_time_ms']
GSR_mean
GSR_variance
HR_mean
HR_variance
75_percentile_GSR
75_percentile_HR
GSRmean_persen_diff
HRmean_persent_diff
GSRme

In [11]:
cols_to_drop = [
    "videoID",
    "start_time",
    "end_time",
    "start_time_sec",
    "end_time_sec",
    "num_samples",
    "start_time_ms",
    "end_time_ms",
    "time"
]

ds = data.drop(columns=cols_to_drop)


In [12]:
for col in ds.columns:
    print(col)

GSR_mean
GSR_variance
HR_mean
HR_variance
75_percentile_GSR
75_percentile_HR
GSRmean_persen_diff
HRmean_persent_diff
GSRmean_diff
HRmean_diff
valence_acc_video
arousal_acc_video
P_id
video_id
Score
valence
arousal
probe
prev_window
pupil_diameter_mean
pupil_diameter_std
diameter_3d_mean
diameter_3d_std
sphere_radius_mean
circle_3d_radius_mean
ellipse_axis_a_mean
ellipse_axis_b_mean
ellipse_angle_mean
ellipse_angle_std
confidence_x_mean
confidence_y_mean
confidence_x_std
confidence_y_std
blink_count
gaze_movement_mean
gaze_movement_std
gaze_dir_variability
eye_center_movement_mean
eye_center_movement_std
missing_value_ratio


In [13]:
ds.shape

(9876, 40)

In [18]:
ds.head()

,GSR_mean,GSR_variance,HR_mean,HR_variance,75_percentile_GSR,75_percentile_HR,GSRmean_persen_diff,HRmean_persent_diff,GSRmean_diff,HRmean_diff,valence_acc_video,arousal_acc_video,P_id,video_id,Score,valence,arousal,probe,prev_window,pupil_diameter_mean,pupil_diameter_std,diameter_3d_mean,diameter_3d_std,sphere_radius_mean,circle_3d_radius_mean,ellipse_axis_a_mean,ellipse_axis_b_mean,ellipse_angle_mean,ellipse_angle_std,confidence_x_mean,confidence_y_mean,confidence_x_std,confidence_y_std,blink_count,gaze_movement_mean,gaze_movement_std,gaze_dir_variability,eye_center_movement_mean,eye_center_movement_std,missing_value_ratio
0,0.425789,0.003011,0.259574,0.001005,0.473684,0.265957,0.267649,0.235091,0.267649,0.235091,1.0,1.0,1,1.0,0.030356,5.000000,5.000000,0.0,1.0,27.022774,2.876173,3.041672,0.305505,10.392305,1.520836,25.568460,27.022774,68.034448,30.662325,0.90293,0.903249,0.108300,0.096104,50.0,24.882023,83.095608,0.076767,1.060841,5.623409,0.033292
1,0.425263,0.002531,0.257660,0.001056,0.447368,0.265957,0.268175,0.237006,0.268175,0.237006,1.0,1.0,1,1.0,0.065719,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662
2,0.517895,0.003336,0.248511,0.001411,0.552632,0.263298,0.175543,0.246155,0.175543,0.246155,1.0,1.0,1,1.0,0.011228,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662
3,0.526316,0.006039,0.265957,0.002856,0.572368,0.297872,0.167122,0.228708,0.167122,0.228708,1.0,1.0,1,1.0,0.276014,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662
4,0.457895,0.002521,0.263404,0.001791,0.500000,0.276596,0.235543,0.231261,0.235543,0.231261,1.0,1.0,1,1.0,0.007019,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662


In [15]:
unique_ids = ds["video_id"].unique()
print(unique_ids)

[1. 2. 3. 4. 5. 6. 7. 8.]


In [19]:
ds1 = ds[ds["video_id"].isin(unique_ids[0:2])]
ds2 = ds[ds["video_id"].isin(unique_ids[2:6])]
ds3 = ds[ds["video_id"].isin(unique_ids[6:8])]
ds1.head()

,GSR_mean,GSR_variance,HR_mean,HR_variance,75_percentile_GSR,75_percentile_HR,GSRmean_persen_diff,HRmean_persent_diff,GSRmean_diff,HRmean_diff,valence_acc_video,arousal_acc_video,P_id,video_id,Score,valence,arousal,probe,prev_window,pupil_diameter_mean,pupil_diameter_std,diameter_3d_mean,diameter_3d_std,sphere_radius_mean,circle_3d_radius_mean,ellipse_axis_a_mean,ellipse_axis_b_mean,ellipse_angle_mean,ellipse_angle_std,confidence_x_mean,confidence_y_mean,confidence_x_std,confidence_y_std,blink_count,gaze_movement_mean,gaze_movement_std,gaze_dir_variability,eye_center_movement_mean,eye_center_movement_std,missing_value_ratio
0,0.425789,0.003011,0.259574,0.001005,0.473684,0.265957,0.267649,0.235091,0.267649,0.235091,1.0,1.0,1,1.0,0.030356,5.000000,5.000000,0.0,1.0,27.022774,2.876173,3.041672,0.305505,10.392305,1.520836,25.568460,27.022774,68.034448,30.662325,0.90293,0.903249,0.108300,0.096104,50.0,24.882023,83.095608,0.076767,1.060841,5.623409,0.033292
1,0.425263,0.002531,0.257660,0.001056,0.447368,0.265957,0.268175,0.237006,0.268175,0.237006,1.0,1.0,1,1.0,0.065719,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662
2,0.517895,0.003336,0.248511,0.001411,0.552632,0.263298,0.175543,0.246155,0.175543,0.246155,1.0,1.0,1,1.0,0.011228,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662
3,0.526316,0.006039,0.265957,0.002856,0.572368,0.297872,0.167122,0.228708,0.167122,0.228708,1.0,1.0,1,1.0,0.276014,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662
4,0.457895,0.002521,0.263404,0.001791,0.500000,0.276596,0.235543,0.231261,0.235543,0.231261,1.0,1.0,1,1.0,0.007019,6.628889,5.786667,0.0,0.0,27.245532,2.059460,3.077961,0.228361,10.392305,1.538981,25.570141,27.245532,53.635414,38.267294,0.94982,0.949588,0.073281,0.061131,11.5,6.114482,30.423697,0.070928,0.215434,2.928634,0.019662


In [ ]:
ds = data[['Score', 'GSRmean_persen_diff', 'HRmean_persen_diff', 'valence_acc_video',
           'arousal_acc_video', 'P_id','video_id','video_flag','label','prev_window2']]